# Hyperparameter Search — Experiment Summary

Reads all training logs from `output/logs/`, parses metrics, and visualizes loss curves interactively.

In [ ]:
log_dir = 'output/logs'

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from datetime import datetime
from IPython.display import display


def parse_log_file(filename):
    with open(filename, 'r') as f:
        content = f.read()

    ts_match = re.search(r'training_log_(\d{8}_\d{6})', filename)
    timestamp = datetime.strptime(ts_match.group(1), '%Y%m%d_%H%M%S') if ts_match else None

    def _get(pattern, cast=str, default=None):
        m = re.search(pattern, content)
        return cast(m.group(1)) if m else default

    return {
        'timestamp':        timestamp,
        'filename':         os.path.basename(filename),
        'num_epochs':       _get(r'NUM_EPOCHS: (\d+)', int),
        'batch_size':       _get(r'BATCH_SIZE: (\d+)', int),
        'learning_rate':    _get(r'LEARNING_RATE: ([\d\.e+-]+)', float),
        'optimizer':        _get(r'OPTIMIZER: (\w+)'),
        'final_train_loss': _get(r'Final Training Loss: ([\d\.]+)', float),
        'final_val_loss':   _get(r'Final Validation Loss: ([\d\.]+)', float),
    }


log_files = sorted(
    [os.path.join(log_dir, f) for f in os.listdir(log_dir)
     if f.startswith('training_log_') and f.endswith('.txt')]
)

records = [parse_log_file(p) for p in log_files]
df = pd.DataFrame(records).sort_values('timestamp', ascending=False).reset_index(drop=True)

pd.set_option('display.max_columns', None)
display(df.drop(columns=['filename']))

## Loss Curves

Select experiments to compare, then click **Update Plots**.

In [ ]:
def extract_loss_history(filepath):
    with open(filepath, 'r') as f:
        content = f.read()
    matches = re.findall(
        r'Epoch \d+/\d+: 100%.*?train_loss=([\d\.]+), val_loss=([\d\.]+)', content
    )
    train_losses = [float(m[0]) for m in matches]
    val_losses   = [float(m[1]) for m in matches]
    return train_losses, val_losses


COLORS = ['#2ecc71','#e74c3c','#3498db','#f1c40f','#9b59b6','#1abc9c','#d35400','#34495e']

checkboxes = {}
for i, row in df.iterrows():
    label = f"{row['optimizer']}  lr={row['learning_rate']}  ({row['filename']})"
    checkboxes[i] = widgets.Checkbox(value=True, description=label,
                                      style={'description_width': 'initial'},
                                      layout={'width': 'auto'})

box = widgets.VBox(
    [widgets.Label('Select experiments:')] + list(checkboxes.values()),
    layout=widgets.Layout(width='100%')
)


def update_plots(*_):
    selected = [i for i, cb in checkboxes.items() if cb.value]
    if not selected:
        print('Select at least one experiment.')
        return

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    for j, idx in enumerate(selected):
        row = df.loc[idx]
        path = os.path.join(log_dir, row['filename'])
        try:
            train_l, val_l = extract_loss_history(path)
            epochs = range(1, len(train_l) + 1)
            label = f"{row['optimizer']} lr={row['learning_rate']}"
            c = COLORS[j % len(COLORS)]
            ax1.plot(epochs, train_l, color=c, label=label, marker='o', markersize=3, alpha=0.8)
            ax2.plot(epochs, val_l,   color=c, label=label, marker='o', markersize=3, alpha=0.8)
        except Exception as e:
            print(f'Error reading {row["filename"]}: {e}')

    for ax, title in [(ax1, 'Training Loss'), (ax2, 'Validation Loss')]:
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.set_yscale('log')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


btn = widgets.Button(description='Update Plots')
btn.on_click(update_plots)

display(box, btn)
update_plots()

## Best Configuration

In [ ]:
best = df.sort_values('final_val_loss').iloc[0]
print(f"Best run:")
print(f"  Optimizer    : {best['optimizer']}")
print(f"  Learning Rate: {best['learning_rate']}")
print(f"  Val Loss     : {best['final_val_loss']:.6f}")

print("\nBest per optimizer:")
display(df.groupby('optimizer')['final_val_loss'].min().sort_values())